# Lexos Classifier Tutorial

This notebook demonstrates the main workflow for using the Lexos `classification` module. Classification tasks are managed through the `Classifier` class, which accepts a list of document data, a list of labels corresponding to the documents, and a pipeline containing the backend logic for performing classification. Lexos comes with two backend pipeline classes `SpaCyTextCategorizerPipeline` and `SklearnClassifierPipeline`.

## Import Data

In the cell below, we will import ten plays by William Shakespeare and convert them to spaCy `Doc` objects. We'll take only the first 90,000 characters in each text so that we don't exceed spaCy's memory limit. We'll also define some labels for use in our classification and check their order against the file names.

In [ ]:
import pandas as pd
import re
from lexos import Loader
from lexos.tokenizer import Tokenizer

path_to_files = "data"

# Load the files
loader = Loader()
loader.load(paths=path_to_files)

# Make sure we don't go over the spaCy memory limit
loader.texts = [t[0:900000] for t in loader.texts[0:10]]
titles = loader.names[0:10]
labels = ['tragedy', 'tragedy', 'comedy', 'history', 'comedy', 'tragedy', 'tragedy', 'comedy', 'history', 'comedy']

# Do some simple preprocessing (lowercasing and removing non-alphabetic characters)
processed_texts = [re.sub(r"[^a-z\s]", "", t.lower()) for t in loader.texts]

# Convert the processed texts into spaCy Docs using Lexos Tokenizer
tokenizer = Tokenizer(model="en_core_web_sm")
docs = list(tokenizer.make_docs(processed_texts))

# Show the loaded files and their labels
df = pd.DataFrame({"name": loader.names[0:10], "label": labels})


## Setup a `Classifier` with the spaCy Backend

We'll use the spaCy backend to illustrate the basic setup. For speed, we'll set the model to make 5 training passes (`epochs`) over the training examples.

In [ ]:
# Import the Classifer and pipeline classes
from lexos.classification import Classifier, SpaCyTextCategorizerPipeline

# Create the pipeline
pipeline = SpaCyTextCategorizerPipeline(epochs=5)

# Initialize the SpaCy classifier with the data, labels, and pipeline
classifier = Classifier(
    data=docs,
    labels=labels,
    titles=titles,
    pipeline=pipeline
)

We now want to split our model into training and test data. We'll reserve 20% of the data for testing with `test_size` and set `random_state` to ensure that we get the same split if we run the cell multiple times.

In [ ]:
# Split the data into training and testing sets using the Classifier
split = classifier.train_test_split(test_size=0.2, random_state=42)

# Show the number of documents in the training and testing sets
print(f"Training Data: {len(split['train']['data'])} docs")
print(f"Testing Data: {len(split['test']['data'])} docs")

Notice that this produces a dictionary containing the data and labels for each set in the split. So you can view the test labels or titles with `split['test']['labels']` and `split['test']['titles']`:

In [ ]:
print(split['test']['labels'])
print(split['test']['titles'])

We're now ready to train ("fit") the classifier.

In [ ]:
# Train the SpaCy classifier on the training data
classifier.fit(
    data=split["train"]["data"],
    labels=split["train"]["labels"],
)

# Make predictions on the test data using the trained SpaCy classifier
predictions = classifier.predict(split["test"]["data"])

# Display the predictions as a pandas DataFrame
pd.DataFrame(
    {
        "Title": split["test"]["titles"],
        "True Label": split["test"]["labels"],
        "Predicted Label": predictions,
    },
)

For the test data we are using in this tutorial, the predictions are not likely to be very accurate (we have few samples per label). We can see the scores used to make the predictions using the `predict_scores` method:

In [ ]:
# Get the scores
scores = classifier.predict_scores()

# Format the scores for easy reading
for i, title in enumerate(split["train"]["titles"]):
    print(f"{title}:")
    for label, score in scores[i].items():
        print(f"  {label}: {score}")

The `evaluate()` method is the simplest way to inspect the accuracy of a trained classifier on the held-out portion of the data.

In [ ]:
classifier.evaluate()

Whilst we may want to improve upon these results before using our model, we can still use it to make predictions on an unseen text (a text outside of our training and test data sets). We can extract a document from a new text using the model and view the model's predictions with the `.cats` (categories) property:

In [ ]:
# Define an unseen text
unseen_text = "history of the novel and literature"

# Get the classification for the unseen text
doc = pipeline.model(unseen_text)
doc.cats

### Multi-Label Modelling

So far, we have used a single-label model in which each document has only one label. However, it is also possible to generate models where each document has multiple labels. Simply define the labels as a list of lists, where each document has a list of labels. We'll need to re-split our dataset from above since we need to assign a new set of labels to the training and test sets.

We are also going to update the pipeline with `score_ranking="document"`. This ranks the scores for each label on a per document basis. Another option is "global". The first mode selects the top N labels for each document from that document’s own score map. The second ranks all class probabilities for the document and then taking the top N classes. This can bias toward the dominant class if one label is consistently stronger in the training set. You'll see the difference if you change the ranking mode and the re-display the predictions. 

In [ ]:
multi_labels = [
    ["tragedy", "revenge"],          # Titus Andronicus
    ["tragedy", "ambition"],         # Antony and Cleopatra
    ["comedy", "love"],              # Measure for Measure
    ["history", "politics"],         # Henry IV Part 1
    ["comedy", "mistaken_identity"], # Two Gentlemen of Verona
    ["tragedy", "love"],             # The Merchant of Venice
    ["history", "power"],            # Pericles
    ["comedy", "marriage"],          # Much Ado About Nothing
    ["history", "politics"],         # Richard III
    ["comedy", "love"],              # The Merry Wives of Windsor
]

pipeline = SpaCyTextCategorizerPipeline(epochs=5, score_ranking="document")

classifier = Classifier(
    data=docs,
    labels=multi_labels,
    titles=titles,
    pipeline=pipeline
)
split = classifier.train_test_split(test_size=0.2, random_state=42)

Now let's train the model.

In [ ]:
classifier.fit()
predictions = classifier.predict(split["test"]["data"])

# Display the predictions as a pandas DataFrame
pd.DataFrame(
    {
        "Title": split["test"]["titles"],
        "True Labels": split["test"]["labels"],
        "Predicted Labels": predictions,
    },
)

You can inspect the scores with `predict_scores` (here we display only the first three documents, along with their titles).

In [ ]:
scores = classifier.predict_scores()[0:3]
for i, score in enumerate(scores[0:3]):
    print(f"{titles[i]}:")
    for k, v in score.items():
        print(f"- {k}: {v}")
    print()

In the cell below, we'll make a prediction on an unseen text (here a summary of part of *A Midsummer Night's Dream*). We'll leave the scores in their original dictionary form.

In [ ]:
text = "The play opens with Theseus and Hippolyta who are four days away from their wedding. Theseus is unhappy about how long he has to wait while Hippolyta thinks it will pass by like a dream. Theseus is confronted by Egeus and his daughter Hermia, who is in love with Lysander, and resistant to her father's demand that she marry Demetrius, whom he has arranged for her to marry. Enraged, Egeus invokes an ancient Athenian law before Duke Theseus, whereby a daughter needs to marry a suitor chosen by her father, or else face death."

doc = pipeline.model(text)
doc.cats


## Use the `scikit-learn` Backend

The the `scikit-learn` backend provides an alternative way of implementing the classifier, giving you access to a number of classification models commonly used in data science. The cell below shows the basic usage.

In [ ]:
from lexos.classification import Classifier, SklearnClassifierPipeline

sklearn_pipeline = SklearnClassifierPipeline()
sklearn_classifier = classifier = Classifier(
    data=docs,
    labels=labels,
    titles=titles,
    pipeline=sklearn_pipeline
)

split2 = sklearn_classifier.train_test_split(test_size=0.33, random_state=42)

sklearn_classifier.fit(
    data=split2["train"]["data"],
    labels=split2["train"]["labels"],
)

predictions = sklearn_classifier.predict(split2["test"]["data"])

# Display the predictions as a pandas DataFrame
pd.DataFrame(
    {
        "Title": split2["test"]["titles"],
        "True Label": split2["test"]["labels"],
        "Predicted Label": predictions,
    },
)


Below is an example with multiple labels:

In [ ]:
sklearn_pipeline = SklearnClassifierPipeline(score_ranking="global")

sklearn_classifier = classifier = Classifier(
    data=docs,
    labels=multi_labels,
    titles=titles,
    pipeline=sklearn_pipeline
)

split2 = sklearn_classifier.train_test_split(test_size=0.33, random_state=42)

sklearn_classifier.fit(
    data=split2["train"]["data"],
    labels=split2["train"]["labels"],
)

predictions = sklearn_classifier.predict(split2["test"]["data"])

# Display the predictions as a pandas DataFrame
pd.DataFrame(
    {
        "Titles": split2["test"]["titles"],
        "True Labels": split2["test"]["labels"],
        "Predicted Labels": predictions,
    },
)

You can use `predict_scores` with the scikit-learn backend. In the cell below, we display on the first three documents so that the output is not too long.

In [ ]:
scores = classifier.predict_scores()[0:3]
for i, score in enumerate(scores[0:3]):
    print(f"{titles[i]}:")
    for k, v in score.items():
        print(f"- {k}: {v}")
    print()

## Use a Precomputed Lexos DTM

Both backends accept a Lexos DTM object where your data has already been tokenized and vectorized. This is useful when you want to reuse a DTM built elsewhere in the workflow or when you want to use a different tokenization strategy than the backend would apply on raw strings.

In [ ]:
from lexos.dtm import DTM

dtm = DTM(docs=docs, labels=titles)

dtm_pipeline = SklearnClassifierPipeline()

dtm_classifier = Classifier(
    data=dtm,
    labels=multi_labels,
    titles=titles,
    pipeline=dtm_pipeline
)

split3 = dtm_classifier.train_test_split(test_size=0.33, random_state=42)

dtm_classifier.fit(
    data=split3["train"]["data"],
    labels=split3["train"]["labels"],
)

predictions = dtm_classifier.predict(split3["test"]["data"])

# Display the predictions as a pandas DataFrame
pd.DataFrame(
    {
        "Titles": split3["test"]["titles"],
        "True Labels": split3["test"]["labels"],
        "Predicted Labels": predictions,
    },
)

## Saving Results

The `classification` module has no built-in method for saving results. Most of the outputs are simple dictionaries, which can be saved in JSON or other formats. The examples in this notebook are frequently pandas DataFrames, which provide a variety of serialization methods. An example of saving result to a CSV file is given below:

In [ ]:
results = pd.DataFrame(
    {
        "Title": split["test"]["titles"],
        "True Label": split["test"]["labels"],
        "Predicted Label": predictions,
    },
)

results.to_csv("classifier_predictions.csv", index=False)

## Saving and Loading Models

The module *does* have methods for saving and reloading trained models. The example below uses the DTM-based model we produced above.

**Important:**

For the spaCy backend, use a directory path like `my_model/`. The scikit-learn backend saves a single `joblib` file. No file extension is not required, but common extensions are `.joblib` and `.pkl`.

In [ ]:
# Configure the path for saving and loading the pipeline
path = "./saved_models/my_pipeline"

# Save the pipeline
dtm_pipeline.save(path)

# Load the pipeline from the saved path
restored = SklearnClassifierPipeline.load(path)

# Use the pipeline to make predictions
print(restored.predict("unseen text"))